# Forecasting Chiller 6 with TTM through `run_recipe`

IBM Granite **TinyTimeMixer** (TTM) is an 805K-parameter time-series foundation model. It is
pretrained on ~700M samples and forecasts a series it has never seen, from context alone.

This notebook drives it entirely through the tsfm tool surface: a catalog card describes the model,
`resolve_model` preflights it, and **`run_recipe` runs it** - the exact same call used for
NaiveForecaster or AutoETS. Only the card changes.

---

### Read this before you run it

**The author could not execute the TTM cells.** The authoring sandbox has no HuggingFace access
(`403` at the proxy on every endpoint) and torch could not be installed there. Rather than ship
invented outputs, **the TTM cells below are shipped unexecuted**. Run them yourself; they should
take about a minute once the weights are cached.

What **was** verified offline and carries real outputs:

* the sktime constructor signature the card targets
* card registration and schema validation
* `resolve_model`'s preflight, including its dependency check
* `training_regime` resolution for every card variant used here
* the classical comparison run

What is **not** verified: the TTM weight download, the fit, and the forecast numbers.

Two things below exist because of live bugs in `resolver.training_regime` - both cards pin
`fit_strategy` and `training_regime` explicitly. Section 6 explains why that is not optional.

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys

need = [p for p in ("torch", "transformers", "accelerate") if importlib.util.find_spec(p) is None]
if need:
    print("installing:", need, "- this pulls the CUDA stack, several GB, expect a few minutes")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "transformers", "accelerate>=0.26.0"])
    print("done - RESTART THE KERNEL, then re-run from here")
else:
    print("torch / transformers / accelerate all present")

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")

SRC = os.path.abspath("src")
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["TSFM_STORE"] = "memory"          # swap for couch to use the real catalog
sys.path.insert(0, SRC)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from servers.tsfm import main as M
from servers.tsfm.io import refs
from servers.tsfm.substrate import resolver as R

print("store:", type(M._STORE).__name__)

## 2. The series

Hourly Chiller 6 load: a daily cycle, a slow drift, light noise. 240 hours of history, forecasting
**6 hours ahead**.

TTM needs a **context window** of past points to condition on. The `granite-timeseries-ttm-r2`
checkpoints come in fixed geometries (context 512 / 1024 / 1536, prediction 96). Our 240 points are
shorter than 512, which matters - see section 6.

In [ ]:
SP, N, FH = 24, 240, [1, 2, 3, 4, 5, 6]
rng = np.random.RandomState(0)
t = np.arange(N)
load = 20 + 4*np.sin(t/SP*2*np.pi) + 0.02*t + rng.normal(0, .3, N)
ref = refs.materialize_iot(load, asset_id="chiller_6")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, load, lw=1); ax.set(title="Chiller 6 load (hourly)", xlabel="hour", ylabel="load")
plt.tight_layout(); plt.show()
print(f"{N} hours of history, forecasting {len(FH)} hours ahead")

## 3. What shape is a recipe?

`recipe_template` is how an agent learns the contract without guessing. The key line is
`estimator_spec`: an estimator is **either** a catalog card (`model_id`) **or** an inline
`sktime_class` + `params`. That seam is what lets one `run_recipe` serve TTM and NaiveForecaster
alike.

In [ ]:
tmpl = M.recipe_template().model_dump()
for line in tmpl["estimator_spec"]:
    print(" -", line[:118])

## 4. The TTM card

A model card is a **pointer**, not weights: `sktime_class` says how to construct it, `params` says
with what, `hf_repo` says where the weights live. Nothing is downloaded at registration.

Verified constructor signature (read from the installed sktime, no torch required):

```
TinyTimeMixerForecaster(model_path='ibm/TTM', revision='main', validation_split=0.2,
                        config=None, training_args=None, compute_metrics=None,
                        callbacks=None, broadcasting=False, use_source_package=False,
                        fit_strategy='minimal')
```

**Note `fit_strategy='minimal'`.** That is sktime's default and it *fine-tunes* - see section 6. We
pin `"zero-shot"` to get an actual zero-shot forecast, and pin `training_regime` so the server does
not have to guess.

In [ ]:
TTM_CARD = {
    "model_id": "ttm_r2_zeroshot",
    "description": ("IBM Granite TinyTimeMixer R2 (805K params, pretrained on ~700M samples). "
                    "True zero-shot: fit_strategy pinned so sktime does not silently fine-tune."),
    "task_ids": ["tsfm_forecasting"],
    "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
    "params": {
        "model_path": "ibm-granite/granite-timeseries-ttm-r2",
        "fit_strategy": "zero-shot",          # sktime's default is "minimal" = fine-tune
    },
    "training_regime": "zero_shot",           # explicit; do not let the server infer it
    "hf_repo": "ibm-granite/granite-timeseries-ttm-r2",
    "model_family": "TinyTimeMixer",
    "framework": "granite-tsfm",
    "provenance": "pretrained",
    "domain": "energy",
    "tags": ["foundation", "zero-shot", "tiny"],
}
d = M.register_model(TTM_CARD).model_dump()
print("registered" if "error" not in d else f"FAILED: {d['error'][:90]}")
print("regime the server will use:", R.training_regime(TTM_CARD))

## 5. Preflight

`resolve_model` checks the class imports **and** that its declared third-party packages are present,
before any download starts. If torch is missing this is where you find out - not 40 seconds into a
backtest.

In [ ]:
d = M.resolve_model("ttm_r2_zeroshot").model_dump()
print("resolvable    :", d.get("resolvable"))
print("regime        :", d.get("training_regime"))
print("weights_from  :", d.get("weights_from"))
print("reason        :", str(d.get("reason"))[:150])

## 6. Why both cards pin `fit_strategy` and `training_regime`

This is not defensive style. Three live bugs in `resolver.training_regime` make the defaults wrong
for TTM specifically, in both directions.

**sktime's TTM defaults to `fit_strategy="minimal"`**, and its own docstring says minimal
*"Fine-tunes only a small subset of the model parameters"*. `_fit` freezes the loaded weights,
unfreezes the mismatched ones, and calls `Trainer(...).train()` - skipping training only when the
checkpoint config matches the requested geometry exactly. Our 240-point series and 6-step horizon do
**not** match a 512/96 checkpoint, so an unpinned card would train.

Meanwhile the server's regime check tests **key presence, never the value**:

```python
_FT_KEYS = ("num_train_epochs","fit_strategy","trainer","fine_tune","finetune","lr")
return "fine_tune" if any(k in params for k in _FT_KEYS) else "zero_shot"
```

Run it on every variant:

In [ ]:
TTM_CLS = "sktime.forecasting.ttm.TinyTimeMixerForecaster"
CASES = [
    ("{} - nothing pinned",                  {"model_path": "x"},                              "minimal fine-tune (sktime default)"),
    ("fit_strategy='zero-shot'",             {"model_path": "x", "fit_strategy": "zero-shot"}, "true zero-shot"),
    ("fit_strategy='full'",                  {"model_path": "x", "fit_strategy": "full"},      "full fine-tune"),
    ("training_args={'num_train_epochs':5}", {"model_path": "x",
                                              "training_args": {"num_train_epochs": 5}},       "fine-tune"),
]
print(f"{'card params':40s} {'server infers':14s} sktime actually does")
for label, p, truth in CASES:
    print(f"  {label:38s} {R.training_regime({'sktime_class': TTM_CLS, 'params': p}):14s} {truth}")
print("")
print("  Our card pins training_regime explicitly, which overrides all of the above.")

Rows 1 and 2 are the problem. **An unpinned card is called `zero_shot` while sktime fine-tunes it**,
and the run record writes `trained: false`. **Pinning `fit_strategy="zero-shot"` - the one thing
that makes it genuinely zero-shot - gets classified `fine_tune`**, because the check sees the key
and never reads the value.

An explicit `training_regime` on the card wins over the inference entirely. That is why it is set.

## 7. Run TTM through `run_recipe`

The whole point. Identical call shape to any other card - only `model_id` differs.

Because the card resolves to `zero_shot`, `run_recipe` takes `_backtest_zero_shot`: set the context
to all-but-the-last-6 points, run inference once, score the held-out tail. **No refit loop** - there
is nothing to fit.

> First run downloads the TTM-R2 weights (~5MB) into `~/.cache/huggingface`.
> **Cells from here to section 9 were not executed by the author.**

In [ ]:
t0 = time.time()
ttm_run = M.run_recipe(
    dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
    recipe={"estimator": {"model_id": "ttm_r2_zeroshot"},
            "fh": FH,
            "eval": {"metrics": ["mape"]}},
).model_dump()
elapsed = time.time() - t0

if "error" in ttm_run:
    print("ERROR:", ttm_run["error"][:300])
else:
    print(f"run_id         : {ttm_run['run_id']}")
    print(f"training_regime: {ttm_run['training_regime']}")
    print(f"metric         : {ttm_run['metric']}")
    print(f"backtest_score : {ttm_run['backtest_score']}")
    print(f"wall clock     : {elapsed:.1f}s")
    print(f"results_file   : {ttm_run['results_file']}")

## 8. The run record

`run_recipe` returns a pointer, not the payload. The record holds the forecast head and the audit.

In [ ]:
if "error" in ttm_run:
    raise SystemExit("TTM did not run, so there is no record to open:\n  "
                     + ttm_run["error"][:220]
                     + "\n\nRun the install cell at the top, restart the kernel, re-run from section 7.")

rec = json.loads(open(ttm_run["results_file"][7:]).read())
for k in ("run_id", "task", "backtest_score", "metric", "training_regime", "trained"):
    print(f"  {k:16s} {rec.get(k)}")
print(f"  forecast_head    {rec.get('forecast_head')}")

fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(t[-72:], load[-72:], lw=1, label="actual")
head = rec.get("forecast_head") or []
if head:
    ax.plot(np.arange(N, N + len(head)), head, "o--", lw=1.5, label="TTM forecast")
ax.axvline(N - 1, color="grey", ls=":"); ax.legend(); ax.set(title="TTM zero-shot, 6h ahead")
plt.tight_layout(); plt.show()

## 9. Is it worth it? Compare honestly

A foundation model's score and a classical model's score are **not comparable by default**, because
`run_recipe` scores them differently: `zero_shot` uses one holdout of `len(fh)` points,
`fit_on_series` averages ~20 expanding folds. Both come back as `backtest_score` under the same
metric name.

To compare like for like, pin the classical cards to `zero_shot` too, so every model is judged on
the **same 6-point holdout**.

In [ ]:
CLASSICAL = {
    "autoreg_zs":    ("sktime.forecasting.auto_reg.AutoREG", {"lags": SP}),
    "theta_zs":      ("sktime.forecasting.theta.ThetaForecaster", {"sp": SP}),
    "naive_seas_zs": ("sktime.forecasting.naive.NaiveForecaster", {"strategy": "last", "sp": SP}),
}
board = {}
for mid, (cls, p) in CLASSICAL.items():
    M.register_model({"model_id": mid, "task_ids": ["tsfm_forecasting"], "provenance": "trained",
                      "description": "classical baseline, pinned to the zero-shot holdout so it is "
                                     "scored identically to the foundation card",
                      "sktime_class": cls, "params": p, "training_regime": "zero_shot"})
    r = M.run_recipe(dataset_path=ref, timestamp_column="timestamp", target_columns=["value"],
                     recipe={"estimator": {"model_id": mid}, "fh": FH,
                             "eval": {"metrics": ["mape"]}}).model_dump()
    if "error" not in r:
        board[mid] = r["backtest_score"]

if "error" not in ttm_run:
    board["ttm_r2_zeroshot"] = ttm_run["backtest_score"]

print("all scored on the SAME 6-point holdout:")
for k, v in sorted(board.items(), key=lambda x: x[1]):
    tag = "  <- foundation, zero-shot" if k.startswith("ttm") else ""
    print(f"   {k:18s} MAPE {v:.4f}{tag}")

### Expect TTM to lose here, and do not hold it against TTM

This series is a clean daily sinusoid with a linear drift, light noise, and **ten full periods** of
history. `AutoREG(lags=24)` identifies that almost exactly in 0.2s. There is nothing left for a
pretrained model to contribute.

Foundation models earn their place where classical models cannot go:

* **Cold start** - a chiller commissioned last week with 30 points of history. AutoREG has nothing
  to fit; TTM does not need anything to fit.
* **Fleet scale** - one model over 5,000 assets, no per-asset parameters to estimate or store.
* **Structure classical models miss** - regime changes, interacting seasonalities, covariates.

If you want the question TTM is actually good at, truncate the history and re-run. The next cell
was executed by the author (TTM errors for want of torch, but the baselines are real), and the
result is sharper than expected:

In [ ]:
SHORT = 40                                  # a newly commissioned asset
ref_short = refs.materialize_iot(load[:SHORT], asset_id="chiller_new")

for mid in ["ttm_r2_zeroshot", "autoreg_zs", "naive_seas_zs"]:
    r = M.run_recipe(dataset_path=ref_short, timestamp_column="timestamp",
                     target_columns=["value"],
                     recipe={"estimator": {"model_id": mid}, "fh": FH,
                             "eval": {"metrics": ["mape"]}}).model_dump()
    out = f"MAPE {r['backtest_score']:.4f}" if "error" not in r else f"ERROR: {r['error'][:60]}"
    print(f"   {mid:18s} {out}")
print("")
print(f"   {SHORT} points of history, 24 lags to estimate. AutoREG does not degrade here -")
print("   it cannot be fitted at all. That is the cold-start argument, and it needs no torch.")

### What that cell shows

`AutoREG` - the model that **won** on 240 points at MAPE 0.0149 - does not merely lose on 40 points.
It returns *"The model specification cannot be estimated"* and produces nothing. Estimating 24
autoregressive lags from 40 observations is not a hard problem, it is an ill-posed one.

The seasonal naive baseline still answers (MAPE 0.0195), because it estimates nothing.

**This is the gap a foundation model fills.** Not "a bit more accurate on a clean series" - *able to
answer at all when there is not enough history to fit anything.* TTM's 805K parameters were
estimated on ~700M samples elsewhere; it needs context, not a training set. A newly commissioned
chiller has context.

That is the benchmark worth running against your own fleet, and it is not the one this notebook's
main series poses.

## 10. Fine-tuning TTM, when you mean it

Fine-tune by pinning `fit_strategy="full"` **and** `training_regime="fine_tune"`. Now the refit path
is correct: there genuinely is training to do, so the expanding-window backtest is the honest
measurement.

`register_finetuned` records the lineage so the catalog knows what came from what.

In [ ]:
FT_CARD = dict(TTM_CARD)
FT_CARD.update({
    "model_id": "ttm_r2_chiller6_ft",
    "description": "TTM-R2 fine-tuned on Chiller 6 telemetry (fit_strategy=full).",
    "params": {"model_path": "ibm-granite/granite-timeseries-ttm-r2", "fit_strategy": "full"},
    "training_regime": "fine_tune",
    "provenance": "finetuned",
    "base_model_id": "ttm_r2_zeroshot",
})
d = M.register_model(FT_CARD).model_dump()
print("registered:", "error" not in d, "| regime:", R.training_regime(FT_CARD))

lin = M.get_model_lineage("ttm_r2_chiller6_ft").model_dump()
print("lineage    :", lin.get("ancestors"), "root:", lin.get("root"))
print("")
print("NOTE: fine-tuning downloads weights AND trains. Not run here.")

## 11. Write the decision back

In [ ]:
if board:
    winner = min(board, key=board.get)
    M.update_model(winner, {
        "tags": ["forecast", "short-term", "recommended"],
        "description": f"SELECTED for chiller-6 6h forecasting: MAPE {board[winner]:.4f} on a "
                       f"6-point zero-shot holdout, measured against "
                       f"{len(board)-1} alternatives scored identically.",
    })
    for m in board:
        if m != winner:
            M.deprecate_model(m, reason=f"lost the chiller-6 6h bake-off: MAPE {board[m]:.4f} "
                                        f"vs {board[winner]:.4f} for {winner} (same 6-point holdout)")
    print("promoted:", winner)
    live = [x["model_id"] for x in M.find_models(task_id="tsfm_forecasting").model_dump()["models"]]
    print("find_models now returns:", sorted(live))

## Takeaways

1. **TTM runs through the same `run_recipe` as everything else.** The card is the only thing that
   changes. That is the payoff of the `model_id` / `sktime_class` seam in `estimator_spec`.
2. **Pin `fit_strategy` and `training_regime` on every foundation card.** sktime's TTM defaults to
   `fit_strategy="minimal"`, which fine-tunes; the server infers `zero_shot` from an unpinned card
   and writes `trained: false`. Explicit beats inferred, and right now inferred is wrong for TTM in
   both directions.
3. **Do not compare a `zero_shot` score to a `fit_on_series` score.** One holdout versus ~20 folds,
   reported under the same field name. Pin the regime on the baselines to make the comparison mean
   something.
4. **A foundation model is not a free upgrade.** On a clean seasonal series with plenty of history,
   AutoREG wins for 0.2s and no torch. Reach for TTM on cold start, fleet scale, or structure the
   classical models cannot express - and benchmark on the data shape you actually have.